In [11]:
#******************************************************************************************

In [ ]:
# ============================================================
# Cell 1: Imports and Setup
# ============================================================
# Import all the libraries we need
import os
import re
import time
import requests

# Set up API key
API_KEY = "ac4aadab-8d33-45ff-8ca2-666723786cfc"

print("Libraries loaded!")

In [ ]:
# ============================================================
# Cell 2: Define the CardPricer Class
# ============================================================
class CardPricer:
    def __init__(self):
        self.api_key = API_KEY
        self.calls = 0
        print("CardPricer ready!")
    
    def parse_input(self, card_input):
        """Parse the card name and number from input string"""
        parts = card_input.strip().split()
        
        if not parts:
            return card_input, None
        
        last = parts[-1]
        
        # Check if last part has number
        if '/' in last:
            number = last.split('/')[0]
            name = ' '.join(parts[:-1])
            return name, number
        elif re.match(r'^[A-Z]*\d+$', last):
            number = last
            name = ' '.join(parts[:-1]) if len(parts) > 1 else card_input
            return name, number
        
        return card_input, None
    
    def search_card(self, name, number=None):
        """Search for card in Pokemon TCG API"""
        url = "https://api.pokemontcg.io/v2/cards"
        headers = {"X-Api-Key": self.api_key}
        
        # Build search query
        query = f'name:"{name}"'
        if number:
            query += f' AND number:{number}'
        
        params = {"q": query, "pageSize": 5}
        
        # Try a few times if it fails
        for i in range(3):
            try:
                print(f"Searching for {name}...")
                response = requests.get(url, headers=headers, params=params, timeout=20)
                self.calls += 1
                
                if response.status_code == 200:
                    cards = response.json().get("data", [])
                    if cards:
                        return cards[0]
                    else:
                        print("No cards found")
                        return None
                elif response.status_code == 429:
                    print("Rate limit, waiting...")
                    time.sleep(30)
                else:
                    print(f"Error: {response.status_code}")
                    
            except Exception as e:
                print(f"Error: {e}")
                if i < 2:
                    time.sleep(5)
        
        return None
    
    def get_price(self, card_input):
        """Get the price for a card"""
        print(f"\nLooking up: {card_input}")
        
        # Parse the input
        name, number = self.parse_input(card_input)
        print(f"Name: {name}, Number: {number}")
        
        # Search for the card
        card = self.search_card(name, number)
        
        if not card:
            print("Card not found!")
            return None
        
        print(f"Found: {card['name']} #{card['number']}")
        print(f"Set: {card['set']['name']}")
        
        # Try to get price from TCGPlayer
        price = None
        tcg_prices = card.get("tcgplayer", {}).get("prices", {})
        
        if tcg_prices:
            # Check different price types
            for price_type in ["holofoil", "normal", "1stEditionHolofoil"]:
                if price_type in tcg_prices:
                    market = tcg_prices[price_type].get("market")
                    if market:
                        price = market
                        print(f"Price: ${price} (TCGPlayer)")
                        break
        
        # Try Cardmarket if no TCGPlayer price
        if not price:
            cm_prices = card.get("cardmarket", {}).get("prices", {})
            if cm_prices:
                price = cm_prices.get("averageSellPrice")
                if price:
                    print(f"Price: ${price} (Cardmarket)")
        
        if not price:
            print("No price available")
        
        return {
            "name": card['name'],
            "number": card['number'],
            "set": card['set']['name'],
            "price": price
        }

print("Class defined!")

In [ ]:
# ============================================================
# Cell 3: Create Instance
# ============================================================
# Make a new pricer to use
pricer = CardPricer()

In [ ]:
# ============================================================
# Cell 4: Test Single Card
# ============================================================
# Test with one card
result = pricer.get_price("Rayquaza V 194/203")
if result:
    print(f"\nResult: {result['name']} - ${result['price']}")


In [ ]:
# ============================================================
# Cell 5: Test Multiple Cards
# ============================================================
# Test with several cards
test_cards = [
    "Rayquaza V 194/203",
    "Cynthia SV82"

]

results = []
for card in test_cards:
    result = pricer.get_price(card)
    results.append(result)
    print("-" * 50)

print(f"\nTotal API calls: {pricer.calls}")


In [4]:


# ============================================================
# Cell 6: Simple Helper Function
# ============================================================
# Quick function to just get a price
def quick_price(card_name):
    """Quick way to get just the price"""
    p = CardPricer()
    result = p.get_price(card_name)
    if result and result['price']:
        return result['price']
    return None

# Test it
price = quick_price("Charizard ex 006/165")
print(f"Quick lookup result: ${price}" if price else "No price found")

CardPricer ready!

Looking up: Charizard ex 006/165
Name: Charizard ex, Number: 006
Searching for Charizard ex...
Error: HTTPSConnectionPool(host='api.pokemontcg.io', port=443): Read timed out. (read timeout=20)
Searching for Charizard ex...
Error: HTTPSConnectionPool(host='api.pokemontcg.io', port=443): Read timed out. (read timeout=20)
Searching for Charizard ex...
Error: HTTPSConnectionPool(host='api.pokemontcg.io', port=443): Read timed out. (read timeout=20)
Card not found!
No price found


In [3]:
# ============================================================
# Cell 7: View All Results
# ============================================================
# Show summary of all cards we looked up
print("\nSummary of all results:")
print("=" * 60)
for r in results:
    if r:
        status = f"${r['price']}" if r['price'] else "No price"
        print(f"{r['name']} ({r['set']}): {status}")
    else:
        print("Card not found")
print("=" * 60)


Summary of all results:
Card not found
Card not found
Card not found
